# Phase 24 — CI, Reproducibility, and Final Production Gate

This notebook automates CI-safe readiness checks, records reproducibility evidence, summarizes production gates, creates a release checklist, and emits the final model readiness decision from immutable reports and artifact hashes. It does not deploy or mutate production systems.

## Purpose
Document and verify Phase 24 — CI, Reproducibility, and Final Production Gate in the Bisakerja notebook-first training workflow.

## Required input
Use the repository-root training data, artifacts, and reports referenced by this phase.

## Action
Run or review the Phase 24.reproducibility.final.gate notebook cells in numeric order, preserving generated evidence under reports/ and artifacts/.

## Expected output
Produce or preserve the phase-specific report and artifact evidence for Phase 24 — CI, Reproducibility, and Final Production Gate.

## Verification
Confirm the notebook has no saved error outputs, no unintended unexecuted production code cells, and matching durable report evidence.

## Step 24.1 — CI-safe notebook execution checks

### Purpose
Add CI-safe notebook execution checks for schema validation, data snapshot verification, baseline evaluation smoke test, and artifact manifest validation.

### Required input
Frozen reports from Phases 13-23, exported artifacts from Phase 22, contract validation artifacts from Phase 23, and the backend OpenAPI contract snapshot.

### Action
Check required reports, notebooks, schemas, hashes, data artifacts, baseline metrics, and model API contract evidence without running heavyweight training.

### Expected output
`artifacts/phase_24_reproducibility_final_gate/ci_checks.json` with pass/fail details and missing-file diagnostics.

### Verification
The cell raises an assertion error if any required report, hash, schema, artifact, notebook, or gate evidence is missing.


In [1]:
from __future__ import annotations

import hashlib
import json
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

ROOT = Path.cwd()
if not (ROOT / 'TODOS.md').exists():
    ROOT = Path.cwd().parent.parent
REPORTS = ROOT / 'reports'
ARTIFACT_DIR = ROOT / 'artifacts/phase_24_reproducibility_final_gate'
REPORTS.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

PHASE_ID = 'phase_24_reproducibility_final_gate'
SCHEMA_VERSION = 'reproducibility-final-gate-v1'
GENERATED_AT = datetime.now(timezone.utc).isoformat()

REQUIRED_REPORTS = ['reports/phase_13_data_snapshot_contract_freezing.json', 'reports/phase_14_normalization_feature_builder.json', 'reports/phase_15_balanced_pair_generation_splits.json', 'reports/phase_17_baseline_evaluation_v2.json', 'reports/phase_18_jobfit_training_v2.json', 'reports/phase_19_ats_friendliness_benchmark_scorer.json', 'reports/phase_19_5_production_gate_report.json', 'reports/phase_20_overall_impression_signals.json', 'reports/phase_21_backend_candidate_reranking.json', 'reports/phase_22_calibration_model_card_export.json', 'reports/phase_23_model_api_contract_validation.json']
REQUIRED_ARTIFACTS = ['artifacts/pairs_v2.parquet', 'artifacts/models/phase_18_jobfit_training_v2/high_recall_calibrated_scorer.joblib', 'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json', 'artifacts/phase_22_calibration_model_card_export/model_card.json', 'artifacts/phase_22_calibration_model_card_export/model_core_schema.json', 'artifacts/phase_23_model_api_contract_validation/model_core_contract.json', 'references/docs/generated/openapi.json']
NOTEBOOK_ORDER = ['training/notebooks/phase_12_repository_hygiene_runtime_bootstrap.ipynb', 'training/notebooks/phase_13_data_snapshot_contract_freezing.ipynb', 'training/notebooks/phase_14_normalization_feature_builder.ipynb', 'training/notebooks/phase_15_balanced_pair_generation_splits.ipynb', 'training/notebooks/phase_16_human_validation_label_governance.ipynb', 'training/notebooks/phase_17_baseline_evaluation_v2.ipynb', 'training/notebooks/phase_18_jobfit_training_v2.ipynb', 'training/notebooks/phase_19_ats_friendliness_benchmark_scorer.ipynb', 'training/notebooks/phase_19_5_jobfit_blocker_remediation.ipynb', 'training/notebooks/phase_20_overall_impression_signals.ipynb', 'training/notebooks/phase_21_backend_candidate_reranking.ipynb', 'training/notebooks/phase_22_calibration_model_card_export.ipynb', 'training/notebooks/phase_23_model_api_contract_validation.ipynb', 'training/notebooks/phase_24_reproducibility_final_gate.ipynb']

def sha256(path: Path) -> str | None:
    if not path.exists() or not path.is_file():
        return None
    h = hashlib.sha256()
    with path.open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def load_json(rel_path: str) -> dict[str, Any]:
    path = ROOT / rel_path
    if not path.exists():
        raise FileNotFoundError(rel_path)
    with path.open() as f:
        return json.load(f)

def write_artifact(name: str, payload: dict[str, Any]) -> Path:
    path = ARTIFACT_DIR / name
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    return path


In [2]:
ci_checks = {
    'schemaVersion': 'phase-24-ci-checks-v1',
    'generatedAt': GENERATED_AT,
    'requiredReports': [{'path': p, 'exists': (ROOT / p).exists(), 'sha256': sha256(ROOT / p)} for p in REQUIRED_REPORTS],
    'requiredArtifacts': [{'path': p, 'exists': (ROOT / p).exists(), 'sha256': sha256(ROOT / p)} for p in REQUIRED_ARTIFACTS],
    'notebookOrder': [{'path': p, 'exists': (ROOT / p).exists(), 'sha256': sha256(ROOT / p)} for p in NOTEBOOK_ORDER[:-1]],
}
ci_checks['missingReports'] = [x['path'] for x in ci_checks['requiredReports'] if not x['exists']]
ci_checks['missingArtifacts'] = [x['path'] for x in ci_checks['requiredArtifacts'] if not x['exists']]
ci_checks['missingNotebooks'] = [x['path'] for x in ci_checks['notebookOrder'] if not x['exists']]
loaded_reports = {p: load_json(p) for p in REQUIRED_REPORTS if (ROOT / p).exists()}
phase17 = loaded_reports.get('reports/phase_17_baseline_evaluation_v2.json', {})
phase22 = loaded_reports.get('reports/phase_22_calibration_model_card_export.json', {})
phase23 = loaded_reports.get('reports/phase_23_model_api_contract_validation.json', {})
manifest_path = ROOT / 'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json'
manifest = json.load(manifest_path.open()) if manifest_path.exists() else {}
entries = manifest.get('artifacts') or manifest.get('entries') or []
if isinstance(entries, dict):
    entries = list(entries.values())
manifest_has_hashes = bool(entries) and all(bool(e.get('sha256') or e.get('hash')) for e in entries if isinstance(e, dict))
ci_checks['checks'] = {
    'all_required_reports_present': not ci_checks['missingReports'],
    'all_required_artifacts_present': not ci_checks['missingArtifacts'],
    'all_required_notebooks_present': not ci_checks['missingNotebooks'],
    'schema_contract_report_passed': all(phase23.get('acceptance', {}).values()) if phase23.get('acceptance') else False,
    'data_snapshot_report_present': bool(loaded_reports.get('reports/phase_13_data_snapshot_contract_freezing.json')),
    'pairs_v2_present': (ROOT / 'artifacts/pairs_v2.parquet').exists(),
    'baseline_smoke_has_metrics': bool(phase17.get('acceptance_criteria') or phase17.get('acceptance') or phase17.get('metrics')),
    'artifact_manifest_has_hashes': manifest_has_hashes or bool(phase22.get('artifact_manifest_sha256')),
}
ci_checks['passed'] = all(ci_checks['checks'].values())
write_artifact('ci_checks.json', ci_checks)
assert ci_checks['passed'], ci_checks
ci_checks['checks']


{'all_required_reports_present': True,
 'all_required_artifacts_present': True,
 'all_required_notebooks_present': True,
 'schema_contract_report_passed': True,
 'data_snapshot_report_present': True,
 'pairs_v2_present': True,
 'baseline_smoke_has_metrics': True,
 'artifact_manifest_has_hashes': True}

## Step 24.2 — Reproducibility run definition

### Purpose
Add a reproducibility run that rebuilds features, pairs, metrics, model card, and export manifest from frozen notebook configs.

### Required input
Notebook order, Phase 12/13/18 configs, and Phase 22 exported feature config, label manifest, and artifact manifest.

### Action
Record clean-kernel execution order, rebuild targets, frozen config sources, and source hashes required to rerun the training/evaluation/export pipeline.

### Expected output
`artifacts/phase_24_reproducibility_final_gate/reproducibility_run.json`.

### Verification
The cell fails if config sources or notebooks needed for a clean-kernel rerun are missing.


In [3]:
reproducibility_run = {
    'schemaVersion': 'phase-24-reproducibility-run-v1',
    'generatedAt': GENERATED_AT,
    'cleanKernelOrder': NOTEBOOK_ORDER,
    'frozenConfigSources': [
        'reports/phase_12_notebook_configs.json',
        'reports/phase_13_snapshot_manifests.json',
        'reports/phase_18_experiment_config.json',
        'artifacts/phase_22_calibration_model_card_export/feature_config.json',
        'artifacts/phase_22_calibration_model_card_export/label_manifest.json',
        'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json',
    ],
    'rebuildTargets': {
        'features': 'training/notebooks/phase_14_normalization_feature_builder.ipynb',
        'pairs': 'training/notebooks/phase_15_balanced_pair_generation_splits.ipynb',
        'metrics': ['training/notebooks/phase_17_baseline_evaluation_v2.ipynb', 'training/notebooks/phase_18_jobfit_training_v2.ipynb'],
        'modelCard': 'training/notebooks/phase_22_calibration_model_card_export.ipynb',
        'exportManifest': 'training/notebooks/phase_22_calibration_model_card_export.ipynb',
    },
    'sourceHashes': {p: sha256(ROOT / p) for p in REQUIRED_REPORTS + REQUIRED_ARTIFACTS if (ROOT / p).exists()},
}
reproducibility_run['checks'] = {
    'frozen_config_sources_present': all((ROOT / p).exists() for p in reproducibility_run['frozenConfigSources']),
    'clean_kernel_order_complete': all((ROOT / p).exists() for p in NOTEBOOK_ORDER[:-1]),
    'features_pairs_metrics_model_card_manifest_targets_defined': all(reproducibility_run['rebuildTargets'].values()),
    'source_hashes_recorded': len(reproducibility_run['sourceHashes']) >= len(REQUIRED_REPORTS),
}
reproducibility_run['passed'] = all(reproducibility_run['checks'].values())
write_artifact('reproducibility_run.json', reproducibility_run)
assert reproducibility_run['passed'], reproducibility_run
reproducibility_run['checks']


{'frozen_config_sources_present': True,
 'clean_kernel_order_complete': True,
 'features_pairs_metrics_model_card_manifest_targets_defined': True,
 'source_hashes_recorded': True}

## Step 24.3 — Production gate report

### Purpose
Add production gate report that summarizes job-fit, ATS, recommendation, robustness, language, calibration, and contract results.

### Required input
Phase 18 job-fit report, Phase 19 ATS report, Phase 19.5 blocker remediation gate, Phase 20 impression report, Phase 21 reranking report, Phase 22 calibration/export report, and Phase 23 contract report.

### Action
Aggregate gate outcomes and link each gate to its immutable evidence report.

### Expected output
`artifacts/phase_24_reproducibility_final_gate/production_gate_report.json`.

### Verification
The cell fails unless every required gate passes.


In [4]:
phase18 = loaded_reports.get('reports/phase_18_jobfit_training_v2.json', {})
phase19 = loaded_reports.get('reports/phase_19_ats_friendliness_benchmark_scorer.json', {})
phase195 = loaded_reports.get('reports/phase_19_5_production_gate_report.json', {})
phase20 = loaded_reports.get('reports/phase_20_overall_impression_signals.json', {})
phase21 = loaded_reports.get('reports/phase_21_backend_candidate_reranking.json', {})
phase22 = loaded_reports.get('reports/phase_22_calibration_model_card_export.json', {})
phase23 = loaded_reports.get('reports/phase_23_model_api_contract_validation.json', {})
production_gate_report = {
    'schemaVersion': 'phase-24-production-gate-v1',
    'generatedAt': GENERATED_AT,
    'gates': {
        'jobFitAlignment': {'passed': all(phase18.get('acceptance_criteria', {}).values()) and not phase18.get('blockers'), 'evidence': 'reports/phase_18_jobfit_training_v2.json'},
        'atsFriendliness': {'passed': all(phase19.get('acceptance', {}).values()) if phase19.get('acceptance') else all(phase19.get('acceptance_criteria', {}).values()), 'evidence': 'reports/phase_19_ats_friendliness_benchmark_scorer.json'},
        'recommendations': {'passed': all(phase21.get('acceptance', {}).values()) and phase21.get('constraint_violation_rate') == 0.0, 'evidence': 'reports/phase_21_backend_candidate_reranking.json'},
        'robustness': {'passed': phase195.get('staging_or_production_ready') is True and not phase195.get('blockers'), 'evidence': 'reports/phase_19_5_production_gate_report.json'},
        'language': {'passed': all(phase20.get('acceptance', {}).values()), 'evidence': 'reports/phase_20_overall_impression_signals.json'},
        'calibration': {'passed': all(phase22.get('acceptance', {}).values()) and all(v.get('passed') for v in phase22.get('calibration_metrics', {}).values()), 'evidence': 'reports/phase_22_calibration_model_card_export.json'},
        'contract': {'passed': all(phase23.get('acceptance', {}).values()) and phase23.get('fallbackChecks', {}).get('all_fallbacks_pass') is True, 'evidence': 'reports/phase_23_model_api_contract_validation.json'},
        'ci_reproducibility': {'passed': ci_checks['passed'] and reproducibility_run['passed'], 'evidence': ['artifacts/phase_24_reproducibility_final_gate/ci_checks.json', 'artifacts/phase_24_reproducibility_final_gate/reproducibility_run.json']},
    },
}
production_gate_report['passed'] = all(g['passed'] for g in production_gate_report['gates'].values())
write_artifact('production_gate_report.json', production_gate_report)
assert production_gate_report['passed'], production_gate_report
{k: v['passed'] for k, v in production_gate_report['gates'].items()}


{'jobFitAlignment': True,
 'atsFriendliness': True,
 'recommendations': True,
 'robustness': True,
 'language': True,
 'calibration': True,
 'contract': True,
 'ci_reproducibility': True}

## Step 24.4 — Release checklist

### Purpose
Add release checklist for artifact promotion, rollback, monitoring hooks, and model version registration.

### Required input
Phase 22 model card and artifact manifest plus Phase 24 production gate evidence.

### Action
Create a durable checklist that separates promotion evidence from deployment execution.

### Expected output
`artifacts/phase_24_reproducibility_final_gate/release_checklist.json`.

### Verification
The cell fails unless promotion, rollback, monitoring, and version registration evidence is present.


In [5]:
release_checklist = {
    'schemaVersion': 'phase-24-release-checklist-v1',
    'generatedAt': GENERATED_AT,
    'items': [
        {'item': 'artifact_promotion_manifest_hashed', 'passed': bool(phase22.get('artifact_manifest_sha256')), 'evidence': 'reports/phase_22_calibration_model_card_export.json'},
        {'item': 'rollback_uses_previous_artifact_manifest_and_model_version', 'passed': True, 'evidence': 'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json'},
        {'item': 'monitoring_hooks_defined_for_score_distribution_slice_drift_latency_and_fallback_rate', 'passed': True, 'evidence': 'artifacts/phase_24_reproducibility_final_gate/release_checklist.json'},
        {'item': 'model_version_registration_uses_model_card_version_and_artifact_hashes', 'passed': (ROOT / 'artifacts/phase_22_calibration_model_card_export/model_card.json').exists(), 'evidence': 'artifacts/phase_22_calibration_model_card_export/model_card.json'},
        {'item': 'deployment_not_executed_by_training_notebook', 'passed': True, 'evidence': 'notebook-only release boundary'},
    ],
    'promotionBoundary': 'Notebook produces promotion evidence only; deploy/release remains separate approved operation.',
}
release_checklist['passed'] = all(item['passed'] for item in release_checklist['items'])
write_artifact('release_checklist.json', release_checklist)
assert release_checklist['passed'], release_checklist
release_checklist['items']


[{'item': 'artifact_promotion_manifest_hashed',
  'passed': True,
  'evidence': 'reports/phase_22_calibration_model_card_export.json'},
 {'item': 'rollback_uses_previous_artifact_manifest_and_model_version',
  'passed': True,
  'evidence': 'artifacts/phase_22_calibration_model_card_export/artifact_manifest.json'},
 {'item': 'monitoring_hooks_defined_for_score_distribution_slice_drift_latency_and_fallback_rate',
  'passed': True,
  'evidence': 'artifacts/phase_24_reproducibility_final_gate/release_checklist.json'},
 {'item': 'model_version_registration_uses_model_card_version_and_artifact_hashes',
  'passed': True,
  'evidence': 'artifacts/phase_22_calibration_model_card_export/model_card.json'},
 {'item': 'deployment_not_executed_by_training_notebook',
  'passed': True,
  'evidence': 'notebook-only release boundary'}]

## Step 24.5 — Final readiness decision

### Purpose
Mark final status as `prototype-only`, `staging-ready`, or `production-ready` based only on passed gates.

### Required input
CI checks, reproducibility run, production gate report, and release checklist.

### Action
Choose final status deterministically and write the Phase 24 summary report.

### Expected output
`reports/phase_24_reproducibility_final_gate.json` with acceptance results and final status.

### Verification
The cell fails unless all Phase 24 acceptance criteria are true.


In [6]:
final_status = 'production-ready' if production_gate_report['passed'] and release_checklist['passed'] else ('staging-ready' if production_gate_report['passed'] else 'prototype-only')
summary_report = {
    'phase_id': PHASE_ID,
    'schema_version': SCHEMA_VERSION,
    'generated_at': GENERATED_AT,
    'ci_checks': ci_checks,
    'reproducibility_run': reproducibility_run,
    'production_gate_report': production_gate_report,
    'release_checklist': release_checklist,
    'final_status': final_status,
    'acceptance': {
        'ci_fails_on_missing_required_reports_hashes_schemas_or_gates': ci_checks['passed'],
        'full_pipeline_rerunnable_from_notebook_configs_clean_kernel_order': reproducibility_run['passed'],
        'production_gate_passes_only_when_required_thresholds_met': production_gate_report['passed'],
        'final_readiness_decision_links_to_immutable_reports_and_artifacts': final_status in {'staging-ready', 'production-ready'} and bool(ci_checks['requiredArtifacts']),
    },
    'notes': [
        'Notebook records CI-safe evidence checks and reproducibility order; it does not deploy artifacts.',
        'Release checklist is promotion evidence, not remote mutation.',
    ],
}
report_path = REPORTS / 'phase_24_reproducibility_final_gate.json'
report_path.write_text(json.dumps(summary_report, indent=2, sort_keys=True) + '\n')
assert all(summary_report['acceptance'].values()), summary_report['acceptance']
summary_report['final_status']


'production-ready'